---
# Homework 2: Convolutional Neural Networks (CNNs) (100 points)


In this homework, we're going to build a convolutional neural network (CNN) architecture and then train it on an image classification dataset (the same dataset as in homework 1).

In [ ]:
# Import core PyTorch modules for building and training neural networks.
# nn contains neural-network layers and loss functions.
import torch
from torch import nn

# Import torchvision utilities for image transforms.
import torchvision
import torchvision.transforms as transforms

# Import Dataset and DataLoader for custom dataset creation and mini-batch loading.
from torch.utils.data import Dataset, DataLoader

# Import plotting, metrics, data handling, and timing utilities used in the notebook.
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import time
import copy

# Import the course helper function that locates packaged data assets.
from mads.lib.path import assets

# Limit CPU thread usage for more stable execution in the course environment.
torch.set_num_threads(4)
torch.set_num_interop_threads(4)


# Loading Data (0 points)

This is an example of building a torch data loader for a dataset. Here we have used it to load the Fashion MNIST image dataset. No code needed here.

The raw data is retrieved from: [Kaggle - Fashion MNIST Dataset](https://www.kaggle.com/datasets/zalando-research/fashionmnist?resource=download)

We have already built a set of training, validation and test data for this homework.

In [ ]:
# Locate the pre-split Fashion-MNIST CSV files provided with the assignment.
train_file = assets.find("fashion_mnist/train.csv")
test_file = assets.find("fashion_mnist/test.csv")
valid_file = assets.find("fashion_mnist/val.csv")

# Read each CSV file into a pandas DataFrame.
train_csv = pd.read_csv(train_file)
test_csv = pd.read_csv(test_file)
valid_csv = pd.read_csv(valid_file)


In [ ]:
class FashionDataset(Dataset):
    """Custom PyTorch Dataset for Fashion-MNIST images stored in a CSV file."""

    def __init__(self, data, transform = None):
        """Store the raw data and prepare separate image and label arrays.""" 
        # Convert the DataFrame to a list of row values for easier processing.
        self.fashion_MNIST = list(data.values)

        # Save any optional transform (here we use ToTensor()).
        self.transform = transform

        # Create containers for labels and flattened pixel arrays.
        label = []
        image = []

        for i in self.fashion_MNIST:
             # The first column contains the class label.
            label.append(i[0])

            # The remaining columns contain the 28x28 image pixels.
            image.append(i[1:])

        # Convert labels to a NumPy array.
        self.labels = np.asarray(label)

        # Reshape flat pixel arrays into 28x28x1 grayscale images and cast to float32.
        self.images = np.asarray(image).reshape(-1, 28, 28, 1).astype('float32')

    def __getitem__(self, index):
        """Return one image-label pair at the requested index."""
        label = self.labels[index]
        image = self.images[index]

        # Apply the transform if one was provided.
        if self.transform is not None:
            image = self.transform(image)

        return image, label

    def __len__(self):
        """Return the total number of examples in the dataset."""
        return len(self.images)


In [ ]:
# Set the mini-batch size used by each DataLoader.
batch_size=256

# Wrap each split in the custom Dataset class and convert images to PyTorch tensors.
train_set = FashionDataset(train_csv, transform=transforms.Compose([transforms.ToTensor()]))
val_set = FashionDataset(valid_csv, transform=transforms.Compose([transforms.ToTensor()]))
test_set = FashionDataset(test_csv, transform=transforms.Compose([transforms.ToTensor()]))

# Create DataLoaders so the model can iterate over the data in batches.
train_loader = DataLoader(train_set, batch_size=batch_size)
val_loader = DataLoader(val_set, batch_size=batch_size)
test_loader = DataLoader(test_set, batch_size=batch_size)


# Question 1: Build the CNN (50 pts)

In this part, you will build a CNN Model.

The model must adhere to the following requirments:
- At least ***two*** 2d convulational layers (https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)
- At least ***two*** AvgPool2d layers (https://pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html)
- At least ***two*** hidden fully connected layers (https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)

The input to the model initialization includes the sizes for two hidden layers and the output channels for two convolutional layers; use these parameters within the layers of your model architecture so they can be set accordingly when initializing the model: <br> 
 - e.g.  `  model = CNN(hidden_1=80,hidden_2=84,output_channel_1=4,output_channel_2=12) ` 

<br>
Additional Notes:

- To improve model performance, you may consider adding batch normalization ([Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift](https://arxiv.org/abs/1502.03167)) after each convolutional layer; addtionally, you may want to add dropout layers to your network and experiment with different activation functions.

- We are suggesting you put all of your modules inside the nn.Sequential function, so it will be easy to "forward" the input. 

- The model weight initialization is provided for you via Xavier uniform initialization for the linear layers and convolutional layers.

- Throughout the notebook, provide your code as instructed only where indicated as below:  

   
   \# YOUR CODE HERE
  
<br>

Feel free to reference the original LeNet paper for inspiration for your model architecture, which was one of the foundational models for early CNN architectures ([Gradient-based learning applied to document recognition](https://ieeexplore.ieee.org/document/726791)).

In [ ]:
class CNN(nn.Module):
    def __init__(self,hidden_1=80,hidden_2=84,output_channel_1=4,output_channel_2=12):
        # Call the parent class constructor.
        super(CNN, self).__init__()

        # Build the CNN as a sequential stack of layers.
        self.net=nn.Sequential(
            # First convolution block:
            # Learn low-level spatial features from the 1-channel input image.
            nn.Conv2d(in_channels=1, out_channels=output_channel_1, kernel_size=5, stride=1, padding=2),

            # Normalize activations to help training stability.
            nn.BatchNorm2d(output_channel_1),

            # Add nonlinearity so the model can learn complex patterns.
            nn.ReLU(),

            # Downsample the feature maps by averaging over 2x2 regions.
            nn.AvgPool2d(kernel_size=2, stride=2),

            # Second convolution block:
            # Learn higher-level features from the first block's outputs.
            nn.Conv2d(in_channels=output_channel_1, out_channels=output_channel_2, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm2d(output_channel_2),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2),

            # Flatten the 3D feature maps into a 1D vector before the dense layers.
            nn.Flatten(),

            # First fully connected hidden layer.
            nn.Linear(output_channel_2 * 7 * 7, hidden_1),
            nn.ReLU(),

            # Second fully connected hidden layer.
            nn.Linear(hidden_1, hidden_2),
            nn.ReLU(),

            # Output layer with 10 logits, one for each Fashion-MNIST class.
            nn.Linear(hidden_2, 10)
        )

        # Initialize convolution and linear layer weights with Xavier initialization
        # to improve training behavior.
        try:
            for m in self.net:
                if type(m) == nn.Linear or type(m) == nn.Conv2d:
                    nn.init.xavier_uniform_(m.weight)
        except:
            for m in self.modules():
                if type(m) == nn.Linear or type(m) == nn.Conv2d:
                    nn.init.xavier_uniform_(m.weight)

    def forward(self, X):
        # Define the forward pass: send the input through the full network.
        return self.net(X)


In [ ]:
# Set model hyperparameters to search over.
# The assignment tunes the number of output channels in the convolutional layers.
# If performance is too low, a smaller learning rate such as 1e-2 or 1e-3 can help.
lr=1*10**-1
hidden_1=80
hidden_2=84
output_channel_1s=[4,6]
output_channel_2s=[12,16]

# Use cross-entropy loss for multi-class classification.
loss_function = nn.CrossEntropyLoss()


In [ ]:
# Define a helper function to evaluate model accuracy on a validation or test loader.
def eval_model(model,data_loader):
    # Switch the model to evaluation mode so layers like batch norm behave correctly.
    model.eval()

    # Store the true labels and predicted labels for all batches.
    y_true_list=[]
    y_pred_list=[]

    model.eval()
    for x,y in data_loader:
        # Run a forward pass to get class scores.
        outputs=model(x)

        # Take the class with the highest score as the prediction.
        _, y_pred = torch.max(outputs, 1)

        # Move results into Python lists for metric calculation.
        y_pred_list.extend(y_pred.clone().detach().tolist())
        y_true_list.extend(y.clone().detach().tolist())

    # Compute classification accuracy from the collected predictions.
    acc=classification_report(y_true_list, y_pred_list,output_dict=True)['accuracy']
    return acc


In [ ]:
# Hidden tests in this cell

# Question 2: Train the CNN model (50 pts)

In this question, you will write a few lines to train the CNN you just built.

This is similar to Homework 1. 

Here's the list of things that you need to implement. All of them can (should) be done using one line of code.

    Initialize the model with a set of hyperparameters
    Initialize the optimizer with the model's trainable parameters
    Set the model into the training mode
    For every batch of data: 
        zero the gradient in the optimizer
        feed the input into the model
        compute the loss
        back propagate the loss
        update the optimizer

* Every 5 epochs evaluate your model on the test set
    * Keep track of the highest accuracy achieved on the test set using variable `current_best`
    * Your best performing model should be stored in variable ```best_model``` (**note**: you will want to use ```copy.deepcopy``` when copying your best model to avoid referencing the wrong model object)

        
Your model should obtain a test set accuracy of at least 0.85 in order to secure full points. A training procedure that is correct but yields an accuracy lower than 0.85 will receive 25 points. 

In [ ]:
# Set a random seed so results are reproducible across runs.
random_seed = 3407
torch.manual_seed(random_seed)

# Force deterministic behavior in cuDNN operations when possible.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [ ]:
# Record the starting time so we can stop if training runs too long.
start_time=time.time()

# Track the best validation accuracy seen so far.
current_best=0

# Store a copy of the best-performing model.
best_model=None

# Try each combination of convolutional output channels.
for output_channel_1 in output_channel_1s:
    for output_channel_2 in output_channel_2s:
        # Initialize the CNN with the current hyperparameter combination.
        model = CNN(hidden_1=hidden_1,
                    hidden_2=hidden_2,
                    output_channel_1=output_channel_1,
                    output_channel_2=output_channel_2)

        # Create the optimizer using the model's trainable parameters.
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)

        # Train for up to 20 epochs for this hyperparameter setting.
        for i in range(20):
            # Stop early if we already reached the target accuracy
            # or if the notebook has been running for too long.
            if current_best >= 0.85 or ((time.time()-start_time)/60) >15:
                break

            # Put the model in training mode.
            model.train()

            # Loop through each mini-batch from the training set.
            for x,y in train_loader:
                # Clear old gradients from the previous optimization step.
                optimizer.zero_grad()

                # Run the inputs through the network to get predicted logits.
                outputs = model(x)

                # Compute the training loss for this batch.
                loss = loss_function(outputs, y)

                # Backpropagate to compute gradients.
                loss.backward()

                # Update model weights using the optimizer.
                optimizer.step()

            # Evaluate every 5 epochs to monitor validation performance.
            if (i + 1) % 5 == 0:
                val_acc = eval_model(model, val_loader)

                # If this model is the best so far, save its accuracy and weights.
                if val_acc > current_best:
                    current_best = val_acc
                    best_model = copy.deepcopy(model)


In [ ]:
# Hidden tests in this cell

In [ ]:
# Hidden tests in this cell